# RAG Pipeline — Test
Embedder → VectorStore → Retriever

Run from `phase-1-fundamentals/` with the `f1-rag` conda env active.

## 1. Embedder

In [ ]:
from embeddings.embedder import Embedder

embedder = Embedder()

In [ ]:
from ingestion.chunker import Chunk

# Embed a single chunk manually
test_chunk = Chunk(chunk_id='test-1', text='Max Verstappen won the 2024 Las Vegas Grand Prix for Red Bull.', metadata={'source': 'test'})
result = embedder.embed([test_chunk])

print(f'Vector length : {len(result[0].embedding)}')
print(f'First 5 values: {result[0].embedding[:5]}')

In [ ]:
# Embed multiple chunks at once — batch encoding
chunks = [
    Chunk(chunk_id='c1', text='Lewis Hamilton won the 2024 British Grand Prix for Mercedes.', metadata={'source': 'test'}),
    Chunk(chunk_id='c2', text='Charles Leclerc took pole position at Monaco 2024 for Ferrari.', metadata={'source': 'test'}),
    Chunk(chunk_id='c3', text='Lando Norris secured McLaren their first win of 2024 in Miami.', metadata={'source': 'test'}),
]
embedded = embedder.embed(chunks)
print(f'Embedded {len(embedded)} chunks')
for e in embedded:
    print(f'  {e.chunk_id}: vector length={len(e.embedding)}')

## 2. VectorStore

In [ ]:
from retrieval.vector_store import VectorStore

vs = VectorStore()
print(f'Chunks already in store: {vs.count()}')

In [ ]:
# Add the embedded chunks — upsert so safe to re-run
vs.add(embedded)
print(f'Chunks after add: {vs.count()}')

In [ ]:
# Query directly with a vector — top 2 results
query_vec = embedded[0].embedding
results = vs.query(query_vec, top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

## 3. Retriever

In [ ]:
from retrieval.retriever import Retriever

# Reuse the same embedder and vs — no reloading
retriever = Retriever(embedder, vs)

In [ ]:
# Ask a question in plain text
results = retriever.retrieve('Which driver won in Britain?', top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

In [ ]:
# Try a different query — should surface Leclerc/Monaco
results = retriever.retrieve('Who was fastest in qualifying at Monaco?', top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

In [ ]:
# Your turn — try your own query below
results = retriever.retrieve('McLaren 2024 win', top_k=3)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')